# COMPASS preprocessing

Stage 0-3 of the COMPASS survival pipeline: schema audit (profile_data only),
cohort compile, longitudinal lab preprocessing, prediction-input build, and
cohort diagnostics. Univariate/multivariate modeling live in
`02_univariate.ipynb` / `03_multivariate.ipynb` and only read the
`prediction_inputs_<arm>/` files this notebook writes. All stages use the
merged `profile_data` parquets.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc", "avpc")
COHORTS = (
    "all",
    "metastatic",          # retrospective ADT-intent strata
    "localized",
    "llm_metastatic",      # LLM stage-pipeline metastatic status
    "llm_nonmetastatic",
)
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start,
# who were presumably androgen-deprived elsewhere first.
EXCLUSIONS = ("none", "pre_adt_castrate")
# Stage 3 writes one independent tree per cohort x endpoint. Each build applies
# only its own time-validity gate: t_platinum never filters NEPC/AVPC and
# t_nepc/t_avpc never filter platinum. The metastatic/localized cohorts are the
# medication-derived ADT-intent strata, applied as an MRN restriction at Stage 3
# (see Stage 1b); "all" is the unrestricted ADT cohort. Stages 0-2 are shared and
# run once per treatment anchor.

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS)


## Stage 0 -- schema audit

Fails fast if a required column is absent or all-null in the merged sources.

In [ ]:
cp.audit_schema()

## Stage 1 -- compile COMPASS cohort data

Endpoint-independent: writes one survival cohort carrying the three modeled endpoints: `PLATINUM`/`TT_PLATINUM`, `NEPC`/`TT_NEPC`, and `AVPC`/`TT_AVPC`. NEPC and AVPC come from separate components of the same LLM criteria-timeline labels (`LLM_annotations/LLM_avpc_nepc_timeline/avpc_nepc_labels.parquet`). Joint `AVPC_NEPC` fields may remain in the cohort as audit metadata, but the union is not modeled as an endpoint. If the label file is not mounted, the stage still succeeds without NEPC/AVPC columns and leaves the platinum pipeline unaffected.

Read the printed summary before spending modelling effort: it reports each endpoint's positive count, provenance breakdowns, and how many events are prevalent (at or before the anchor) and will therefore be excluded by the incident-endpoint filter.

In [ ]:
cp.compile_cohort(arms=ARMS)

## Stage 1b -- medication-derived ADT-intent strata

Writes a combined audit label, one MRN list per stratum, and preliminary
incident endpoint counts under `<data_root>/mrn_lists/`. Stage 3 consumes those
MRN lists via `--restrict-to-mrns` to build the `metastatic` and `localized`
cohort trees; the count table is an early warning for cohort/endpoint pairs with
too few events to fit reliably.

**Interpretation warning:** `ADT_INTENT` uses the full observed ADT course
(duration, cessation/restart, and later definitive escalation). Platinum is
excluded from the classifier, but the stratum is still not available
prospectively at the landmark. Cohort-stratified results are retrospective.


In [ ]:
STRATUM_FILES = cp.build_adt_intent_mrn_lists()

import pandas as pd

preliminary_counts = pd.read_csv(STRATUM_FILES["counts"])
preliminary_counts


## Stage 1c -- LLM-adjudicated metastatic strata

Splits the same ADT cohort on the `met_diagnosis` LLM pipeline's
`met_dx_labels.parquet` instead of on medication history. That task is a
purpose-built prostate metastatic-diagnosis extraction: it is prostate-specific
by construction, runs a deterministic veto gate that rejects negated and hedged
mentions, and excludes regional pelvic nodal (N1) disease -- which the generic
`cancer_stage` extraction's `metastatic_sites` would have counted as
metastatic. It also materializes patients with no qualifying evidence as
explicit negatives, so `llm_nonmetastatic` is a real label rather than an
absence.

A patient is `llm_metastatic` if `has_metastatic_disease` is true, and
`llm_nonmetastatic` if it is false; together the two strata partition the
labelled cohort.

**Coverage:** ADT-cohort patients missing from the labels file are treated as
UNLABELLED and dropped from both strata, with the count and coverage fraction
printed. They are *not* folded into `llm_nonmetastatic`: the upstream task
writes its own auto-negatives, so absence means the labels file does not cover
this cohort (an `--mrns`-limited run, or a cohort rebuilt after the LLM run),
and calling them negative would assert a verdict the LLM never made.

Consequence: the two LLM strata together cover slightly fewer patients than the
ADT-intent strata do, so **their Ns are not directly comparable** -- check the
printed coverage before reading across cohorts. Pass
`require_full_cohort_coverage=True` to make any gap fatal instead.

**Interpretation warning:** like `ADT_INTENT`, this adjudicates the full
observed record and is a *retrospective* stratum, not a landmark-available
label. `first_metastasis_date` is carried into the audit file so a
baseline-only variant can be cut later without re-running the LLM.

Set `LLM_MET_LABELS_PATH` if the extraction output is mounted elsewhere.

In [ ]:
LLM_MET_FILES = cp.build_llm_met_mrn_lists()

llm_met_labels = pd.read_csv(LLM_MET_FILES["labels"])
llm_met_labels["LLM_METASTATIC"].value_counts(dropna=False)

## Stage 2 -- preprocess raw labs (per arm anchor)

Expensive: full raw lab standardization. The Parquet cache
(`consolidated_longitudinal_data_<arm>.parquet`) makes reruns cheap, but the
first pass may be slow.

In [ ]:
for run in cp.stage2_runs(RUNS):
    cp.preprocess_labs(run)


## Stage 2b -- pre-ADT castrate exclusion list

Must run AFTER Stage 2: it reads `longitudinal_prediction_data_adt.csv`, whose
`LAB_VALUE` is already unit-standardized to ng/dL and whose `t_lab` is signed
days from the ADT anchor.

Flags patients with any testosterone below 50 ng/dL recorded strictly before
their ADT anchor -- presumed androgen-deprived elsewhere first, making their
recorded ADT start a transfer-of-care artifact rather than a treatment origin.
Stage 3 removes these MRNs via `--exclude-mrns` for every `_noprecastrate` run;
the list is cohort-independent and built once.

In [ ]:
PRE_ADT_CASTRATE_FILE = cp.build_pre_adt_castrate_mrn_list()

pre_adt_castrate = pd.read_csv(PRE_ADT_CASTRATE_FILE)
print(f"{len(pre_adt_castrate):,} patients excluded from every _noprecastrate run")
pre_adt_castrate.head()


## Stage 3 -- build prediction inputs + cohort diagnostics

Set `REBUILD_PREDICTION_INPUTS = False` to skip rebuilding. This cell builds both endpoint trees. Platinum requires only valid `t_platinum`; NEPC requires only valid incident `t_nepc`. Shared death/follow-up checks still apply to both.

In [ ]:
REBUILD_PREDICTION_INPUTS = True

for run in RUNS:
    if REBUILD_PREDICTION_INPUTS:
        cp.build_prediction_inputs(run)
    else:
        print(f"[skip] prediction-input rebuild disabled for {run['label']}")
    cp.cohort_diagnostics(run)

## Stage 3b -- sequencing, Gleason, and PRS inputs

Builds `prediction_inputs_<arm>/somatic_gleason/` from the sample-level
somatic matrix published by `PROFILE_data_processing` and the Gleason timeline
published by `LLM_clinical_annotations`. It creates two cohorts: the sequencing
sample closest to ADT start with follow-up from specimen collection, and the
Gleason score closest to ADT start with follow-up from the score date. PRSs use
ADT start itself as the prediction origin.

In [ ]:
for run in RUNS:
    cp.build_somatic_gleason_inputs(run)